# ML Data Cleaning 
## Student Retention Prediction

This notebook cleans the Higher Education Predictors of Student Retention dataset.

**Goal:** Identify high-risk students at the end of their first semester using:
- Socio-economic background
- Enrollment data
- 1st semester academic performance

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO


## 1. Load the Dataset

In [25]:
import subprocess
import sys
import pandas as pd
import os

subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])

import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/higher-education-predictors-of-student-retention")
print("Path to dataset files:", path)


# List files in the downloaded path
print(os.listdir(path))

# Read the dataset
df = pd.read_csv(path + "/dataset.csv")
print(df.head())

Path to dataset files: C:\Users\mtere\.cache\kagglehub\datasets\thedevastator\higher-education-predictors-of-student-retention\versions\2
['dataset.csv']
   Marital status  Application mode  Application order  Course  \
0               1                 8                  5       2   
1               1                 6                  1      11   
2               1                 1                  5       5   
3               1                 8                  2      15   
4               2                12                  1       3   

   Daytime/evening attendance  Previous qualification  Nacionality  \
0                           1                       1            1   
1                           1                       1            1   
2                           1                       1            1   
3                           1                       1            1   
4                           0                       1            1   

   Mother's qualification  F

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 35 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Nacionality                                     4424 non-null   int64  
 7   Mother's qualification                          4424 non-null   int64  
 8   Father's qualification                          4424 non-null   int64  
 9   Mother's occupation                             4424

## 2. Feature Selection

Remove semester 2 variables (data leakage) and keep only relevant features for predicting at the end of semester 1.

**Kept features:**
- **Socio-economic background:** Marital status, Gender, Mother's qualification, Father's qualification, Mother's occupation, Father's occupation, Educational special needs, Nationality, International status
- **Enrollment data:** Application mode, Course, Daytime/evening attendance, Previous qualification, Age at enrollment
- **Financial indicators:** Tuition fees up to date, Debtor, Scholarship holder, Displaced
- **1st semester academic performance:** All curricular units data (credited, enrolled, evaluations, approved, grade, without evaluations)

**Removed features:**
- All semester 2 variables (prevents data leakage)
- Macroeconomic indicators (not essential for this analysis)

In [27]:
columns_to_keep = [
    # Socio-economic & Demographics
    'Marital status',
    'Gender',
    'Nacionality',
    'International',
    'Mother\'s qualification',
    'Father\'s qualification',
    'Mother\'s occupation',
    'Father\'s occupation',
    'Educational special needs',
    
    # Enrollment & Course Information
    'Application mode',
    'Course',
    'Daytime/evening attendance',
    'Previous qualification',
    'Age at enrollment',
    
    # Financial Status
    'Tuition fees up to date',
    'Scholarship holder',
    'Debtor',
    'Displaced',
    
    # 1st Semester Academic Performance
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (without evaluations)',
    
    # Target
    'Target'
]

df = df[columns_to_keep].copy()

print(f"Dataset shape after feature selection: {df.shape}")
print(f"\nFeatures selected: {len(columns_to_keep) - 1}")
df.head()

Dataset shape after feature selection: (4424, 25)

Features selected: 24


,Marital status,Gender,Nacionality,International,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Educational special needs,Application mode,...,Scholarship holder,Debtor,Displaced,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Target
0,1,1,1,0,13,10,6,10,0,8,...,0,0,1,0,0,0,0,0.000000,0,Dropout
1,1,1,1,0,1,3,4,4,0,6,...,0,0,1,0,6,6,6,14.000000,0,Graduate
2,1,1,1,0,22,27,10,10,0,1,...,0,0,1,0,6,0,0,0.000000,0,Dropout
3,1,0,1,0,23,27,6,4,0,8,...,0,0,1,0,6,8,6,13.428571,0,Graduate
4,2,0,1,0,22,28,10,10,0,12,...,0,0,0,0,6,9,5,12.333333,0,Graduate


## 3. Target Variable Encoding

In [28]:
print("Target variable distribution (before encoding):")
print(df['Target'].value_counts())

df['Target'] = df['Target'].map({
    'Dropout': 1,
    'Graduate': 0,
    'Enrolled': 0
})

print("\nTarget variable distribution (after encoding):")
print(df['Target'].value_counts())

Target variable distribution (before encoding):
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target variable distribution (after encoding):
Target
0    3003
1    1421
Name: count, dtype: int64


## 4. Handle Missing Values

In [29]:
df = df[df['Target'].notna()]
df.isnull().sum()

Marital status                                    0
Gender                                            0
Nacionality                                       0
International                                     0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Educational special needs                         0
Application mode                                  0
Course                                            0
Daytime/evening attendance                        0
Previous qualification                            0
Age at enrollment                                 0
Tuition fees up to date                           0
Scholarship holder                                0
Debtor                                            0
Displaced                                         0
Curricular units 1st sem (credited)               0
Curricular u

## 5. Remove Duplicates

In [30]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

Duplicates: 3


## 6. One-Hot Encoding for Categorical Variables

In [31]:
df['Application mode'] = df['Application mode'].astype('category')
df['Previous qualification'] = df['Previous qualification'].astype('category')

df = pd.get_dummies(df, columns=[
    'Application mode',
    'Previous qualification'
], drop_first=True)

## 7. Final Dataset Overview

In [32]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
Index: 4421 entries, 0 to 4423
Data columns (total 56 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4421 non-null   int64  
 1   Gender                                          4421 non-null   int64  
 2   Nacionality                                     4421 non-null   int64  
 3   International                                   4421 non-null   int64  
 4   Mother's qualification                          4421 non-null   int64  
 5   Father's qualification                          4421 non-null   int64  
 6   Mother's occupation                             4421 non-null   int64  
 7   Father's occupation                             4421 non-null   int64  
 8   Educational special needs                       4421 non-null   int64  
 9   Course                                          4421 non-

,Marital status,Gender,Nacionality,International,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Educational special needs,Course,...,Scholarship holder,Debtor,Displaced,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Target
count,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,...,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000,4421.000000
mean,1.178693,0.351504,1.254694,0.024881,12.324587,16.456458,7.318706,7.820403,0.011536,9.901832,...,0.248360,0.113775,0.548292,0.710473,6.273468,8.302873,4.708437,10.644876,0.137752,0.321194
std,0.605935,0.477494,1.749028,0.155781,9.025444,11.045493,3.998301,4.857503,0.106796,4.329563,...,0.432111,0.317574,0.497719,2.361235,2.477426,4.176790,3.093607,4.839752,0.691105,0.466988
min,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,1.000000,0.000000,2.000000,3.000000,5.000000,5.000000,0.000000,6.000000,...,0.000000,0.000000,0.000000,0.000000,5.000000,6.000000,3.000000,11.000000,0.000000,0.000000
50%,1.000000,0.000000,1.000000,0.000000,13.000000,14.000000,6.000000,8.000000,0.000000,10.000000,...,0.000000,0.000000,1.000000,0.000000,6.000000,8.000000,5.000000,12.285714,0.000000,0.000000
75%,1.000000,1.000000,1.000000,0.000000,22.000000,27.000000,10.000000,10.000000,0.000000,13.000000,...,0.000000,0.000000,1.000000,0.000000,7.000000,10.000000,6.000000,13.400000,0.000000,1.000000
max,6.000000,1.000000,21.000000,1.000000,29.000000,34.000000,32.000000,46.000000,1.000000,17.000000,...,1.000000,1.000000,1.000000,20.000000,26.000000,45.000000,26.000000,18.875000,12.000000,1.000000


## 8. Save Cleaned Dataset

In [34]:
# Save your cleaned dataset into that folder
df.to_csv("datasets/clean_dataset.csv", index=False)
print("Saved!")

Saved!
